In [1]:
import os
from google.cloud import bigquery
from dotenv import load_dotenv
import vertexai
from google.cloud import discoveryengine_v1 as discoveryengine
from vertexai import generative_models as genai  # Añadir esta línea
from vertexai.generative_models import (
    FunctionDeclaration,
    GenerationConfig,
    Tool,
)

In [2]:
load_dotenv()  # Carga las variables desde .env al entorno
client = bigquery.Client(project='dataton-2024-team-01-cofares')
# Ahora puedes acceder a las variables de entorno
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")

In [3]:
# Configuración del cliente de Vertex AI
PROJECT_ID = "dataton-2024-team-01-cofares"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
#multimodal_model = GenerativeModel("gemini-1.5-flash-001")

# Inicializa el cliente de Discovery Engine
discovery_client = discoveryengine.RankServiceClient()  

In [4]:

def get_products(prompt):
    client = bigquery.Client(project=project_id)
    query = """
    WITH QueryEmbedding AS (
      SELECT
        ml_generate_embedding_result AS query_embedding
      FROM
        ML.GENERATE_EMBEDDING(
          MODEL `dataton-2024-team-01-cofares.datos_cofares.text_embedding`,
          (SELECT @prompt AS content),  -- Aquí usamos el parámetro
          STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    )
    SELECT
      d.nombre_completo_material AS nombre,
      d.txt_mas_informacion_del_producto AS descripcion,
      d.txt_instrucciones_de_uso AS modo_implementacion,
      d.codigo_web,
      d.URI_primera_imagen,
      d.codigo_nacional,
      ML.DISTANCE(
        qe.query_embedding,
        e.ml_generate_embedding_result,
        'COSINE'
      ) AS distance_to_query
    FROM
      `dataton-2024-team-01-cofares.datos_cofares.data_final_temp` AS d
    INNER JOIN
      `dataton-2024-team-01-cofares.datos_cofares.SalidaEmbeddings_temp` AS e
      ON d.codigo_web = e.title
    INNER JOIN QueryEmbedding AS qe
      ON TRUE
    ORDER BY
      distance_to_query
    LIMIT 10;
    """.format(prompt)
    # Configura el parámetro para el prompt
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("prompt", "STRING", prompt)
        ]
    )

    query_job = client.query(query, job_config=job_config)
    results = query_job.result()
    
    products = []
    for row in results:

        descripcion = row.descripcion
        if not row.descripcion:
            descripcion = '-'
        
        modo_implementacion = row.modo_implementacion
        if not row.modo_implementacion:
            modo_implementacion = '-'


        # Cambia la URL si es necesario
        imagen_url = row.URI_primera_imagen 
        if imagen_url and imagen_url.startswith('gs:/'):
            imagen_url = imagen_url.replace('gs://dataton-2024-team-01-cofares-datastore/imagenes/', 'https://storage.googleapis.com/dataton-2024-team-01-cofares-datastore/imagenes/reto_cofares/')
        products.append({
            "codigo_web": row.codigo_web,
            "nombre": row.nombre,
            "codigo_nacional": row.codigo_nacional,
            "descripcion": descripcion,
            "modo_implementacion": modo_implementacion,
            "imagen_url": imagen_url,
            "distance_to_query": row.distance_to_query
        })
    return products

In [5]:
def rerank_products(prompt, products):
    ranking_config = discovery_client.ranking_config_path(
        project=PROJECT_ID,
        location=LOCATION,
        ranking_config="default_ranking_config",
    )
    
    records = [
        discoveryengine.RankingRecord(
            id=str(index),
            title=product["nombre"],
            content=product["descripcion"] + " " + product["modo_implementacion"]
        )
        for index, product in enumerate(products)
    ]
    
    request = discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model="semantic-ranker-512@latest",
        top_n=10, # cantidad de productos a rankear
        query=prompt,
        records=records,
    )
    
    response = discovery_client.rank(request=request)
    
    # Aseguramos que los productos están formateados según el esquema
    ranked_products = [
        {
            "codigo_web": products[int(record.id)]["codigo_web"],
            "nombre": products[int(record.id)]["nombre"],
            "codigo_nacional": products[int(record.id)]["codigo_nacional"],
            "descripcion": products[int(record.id)]["descripcion"],
            "modo_implementacion": products[int(record.id)]["modo_implementacion"],
            "imagen_url": products[int(record.id)]["imagen_url"],
            "distance_to_query": products[int(record.id)]["distance_to_query"]
        }
        for record in response.records[:5] # cantidad de productos a mostrar
    ]
    
    return {"products": ranked_products}

In [6]:
#FUNCTION CALLING

# Define el schema
product_schema = FunctionDeclaration(
    name="product_query",
    description="Fetches relevant product information based on a search prompt.",
    parameters={
        "type": "object",
        "properties": {
            "products": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "codigo_web": {"type": "string", "description": "Product web code"},
                        "nombre": {"type": "string", "description": "Product name"},
                        "codigo_nacional": {"type": "string", "description": "National product code"},
                        "descripcion": {"type": "string", "description": "Product description"},
                        "modo_implementacion": {"type": "string", "description": "Mode of implementation"},
                        "imagen_url": {"type": "string", "description": "Image URL"},
                        "distance_to_query": {"type": "number", "description": "Semantic distance to query"}
                    }
                }
            }
        }
    }
)

# Define tools antes de inicializar el modelo
tools = [Tool(function_declarations=[product_schema])]

In [7]:
PROJECT_ID = "dataton-2024-team-01-cofares"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

# Importa el modelo de Gemini Flash 1.5
import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)


# Model definition
multimodal_model = genai.GenerativeModel(
"gemini-1.5-flash",
generation_config=GenerationConfig(temperature=0),
tools=tools)

chat = multimodal_model.start_chat(response_validation=False)

In [8]:
def generate_response(prompt):  # Eliminamos el parámetro products
    #chat = multimodal_model.start_chat()

    instruction_prompt = f"""
    Eres un asistente farmacéutico experto. 
    
    Consulta recibida de un farmacéutico: "{prompt}"
    
    Por favor, responde de la siguiente manera:
    
    - Saluda a los usuarios y pregúntales en qué puedes ayudarles hoy.
    - Resume la petición del usuario y pídale que confirme que ha entendido correctamente.
    - Si es necesario, pida detalles aclaratorios.
    - Utilice ${tools} para recibir un listado de productos rankeados para ayudar al usuario con su tarea.
    - Agradezca al usuario su colaboración y despídase.
    """

    try:
        response = chat.send_message(instruction_prompt)
        response.candidates[0].content.parts[0]
        
        # Verificar si hay una llamada a función
        for candidate in response.candidates:
            for part in candidate.content.parts:
                if hasattr(part, 'function_call') and part.function_call:
                    # Ejecutar búsqueda de productos
                    products = get_products(prompt)
                    if not products:
                        return "Lo siento, no encontré productos que coincidan con tu búsqueda."
                    
                    ranked_products = rerank_products(prompt, products)
                    
                    # Enviar los resultados al modelo para generar una respuesta contextual
                    results_prompt = f"""
                    Basado en la búsqueda "{prompt}", he encontrado estos productos:
                    {[product['nombre'] for product in ranked_products['products']]}
                    
                    Por favor, genera una respuesta útil que:
                    1. Mencione los productos encontrados
                    2. Explique por qué son relevantes
                    3. Proporcione recomendaciones de uso
                    """
                    
                    final_response = chat.send_message(results_prompt)
                    return {
                        "type": "product_search",
                        "message": final_response.text,
                        "products": ranked_products["products"]
                    }
                
        # Si no hay llamada a función, devolver la respuesta conversacional
        return {
            "type": "conversation",
            "message": response.text
        }
                    
    except Exception as e:
        return f"Lo siento, ocurrió un error: {str(e)}"

In [13]:
# Ejemplo de uso
prompt = "Busco una crema para las estrías"

In [14]:

products = get_products(prompt)  # Llamar a la función para obtener productos
# Imprimir los productos obtenidos
print("Productos obtenidos:")
for product in products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos obtenidos:
Nombre: Duplo Farline Crema De Manos Anti_Age, 2 x 50 ml, Descripción: Duplo Farline Crema De Manos Anti-Age ayuda a aclarar las manchas en las manos y contribuye a evitar su aparición. Esta es una crema de acción especial antiedad, cuyos componentes ayudan a difuminar las arrugas y manchas cutáneas: alteraciones de la piel que se presentan en las manos como consecuencia de la exposición solar extrema, así como a causa de la sequedad producida por agentes contaminantes. Al ser aplicada diariamente sobre las manos logra nutrir la piel en profundidad, creando una barrera contra elementos dañinos y rayos solares. Además, es una crema de fácila absorción.Usada regularmente, esta crema de manos ayuda a suavizar la piel y la protege de agentes externos como las radiaciones solares UV, los jabones abrasivos, la sequedad producida por el polvo, el aire frío en invierno y los cambios de temperatura. Entre otros ingredientes contiene aceite de oliva y pantenol, que ejercen u

In [15]:
# Llamar a la función de reranking
ranked_products = rerank_products(prompt, products)["products"]  # Accede a la lista de productos

# Imprimir los productos rankeados
print("Productos rankeados:")
for product in ranked_products:
    print(f"Nombre: {product['nombre']}, Descripción: {product['descripcion']}, Modo de implementación: {product['modo_implementacion']}, Distancia: {product['distance_to_query']}")

Productos rankeados:
Nombre: CREMA ACEITE ROSA MOSQU 50ML, Descripción: "CREMA DE ACEITE ROSA MOSQUETA. Presentado en un envase de 50 ml. Nutre, regenera, repara y protege la piel, dando una sensación inmediata de suavidad. Contribuye a reducir las arrugas. Mejora el estado de las cicatrices. Aporta a la piel la hidratación y emoliencia necesarias para evitar la sequedad cutánea. Protege de agresiones externas como las variaciones de humedad y temperatura ambientales. Favorece la rápida recuperación del equilibrio fisiológico de la piel.  Sin parabenos, no testado en animales.  Precauciones: Evitar contacto con los ojos y mucosas.", Modo de implementación: Aplicar dos veces al día, mañana y noche, después de limpiar y tonificar la piel. Repartir la crema por la cara, cuello y escote dando un suave masaje con las yemas de los dedos hasta su completa absorción., Distancia: 0.3748334345316666
Nombre: Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml, Descripción: Pack Uresim Ser

In [16]:
response_text = generate_response(prompt)  # Generar la respuesta
print(response_text)

{'type': 'product_search', 'message': '¡Hola! He encontrado algunas cremas que podrían ser útiles para las estrías:\n\n* **CREMA ACEITE ROSA MOSQU 50ML:** Esta crema contiene aceite de rosa mosqueta, conocido por sus propiedades regenerativas y cicatrizantes. Puede ayudar a mejorar la apariencia de las estrías, haciéndolas menos visibles. Se recomienda aplicar la crema dos veces al día, masajeando suavemente sobre la zona afectada.\n* **Pack Uresim Serum Ác.Hialurónico 30 ml + Crema Nutri+ 50 ml:** Este pack contiene un serum con ácido hialurónico, que hidrata la piel en profundidad, y una crema nutritiva que ayuda a mejorar la elasticidad de la piel. Ambos productos pueden ser beneficiosos para las estrías, ya que ayudan a mantener la piel hidratada y flexible. Se recomienda aplicar el serum por la mañana y la crema por la noche.\n* **Pack Eucerin Urea Repair Plus Bálsamo Nutritivo + Gel Baño, 450 ml + 400 ml:** Este pack contiene un bálsamo nutritivo y un gel de baño, ambos con urea,